# 🎬 Colab Video Renderer
One-click Remotion render using datacenter CPU + >500Mbps pipe. Upload the companion `colab-render-*.zip` when prompted, then **Run All**.

> Generated by `POST /api/export-colab` — contains identical `VideoComposition` + motion registry as the editor (frame-accurate parity).


In [ ]:
# Step 1: Check Environment (Node, vCPUs, RAM, FFmpeg)
!node -v && echo "---" && echo "vCPUs: $(nproc)" && free -h | grep Mem && echo "---" && ffmpeg -version | head -n1


In [ ]:
# Step 2: Unzip Bundle (upload colab-render-*.zip via Files panel if not present)
!ls -lh colab-render-*.zip 2>/dev/null || echo "Please upload colab-render-*.zip via Files panel, then re-run"
!unzip -q -o colab-render-*.zip 2>/dev/null && echo "✅ Unzipped" || echo "No zip found yet"
!ls -d colab-render-*/ 2>/dev/null | head -n1 || ls -d */ | head


In [ ]:
# Step 2b: Install Dependencies (--prefer-offline leverages Colab cache)
!cd $(ls -d colab-render-*/ 2>/dev/null | head -n1 || echo .) && npm install --no-audit --prefer-offline


In [ ]:
# Step 3: Inspect Bundle & Chunk Plan (project_state + chunk partitioning preview)
BUNDLE_DIR=$(ls -d colab-render-*/ 2>/dev/null | head -n1); BUNDLE_DIR=${BUNDLE_DIR:-.}
!cd $BUNDLE_DIR 2>/dev/null || cd .; echo "Bundle dir: $(pwd)" && ls -lh project_state.json scripts/render_colab.mjs 2>&1 | head && echo "---" && cat project_state.json | python3 -c "import json; s=json.load(open('project_state.json')); print(f\"Duration: {s.get('totalDuration')}s | Frames: {int(s.get('totalDuration',0)*s.get('fps',30))} | Chunks: N=min(6,max(2,cpu))\")" 2>&1 | head


In [ ]:
# Step 3b: Parallel Chunk Render (N workers via Promise.all → FFmpeg concat stitch in <1s)
# Each worker renders frameRange [start,end] to .chunks/chunk_*.mp4 concurrently — 8-10GB RAM & 100% vCPUs
BUNDLE_DIR=$(ls -d colab-render-*/ 2>/dev/null | head -n1); BUNDLE_DIR=${BUNDLE_DIR:-.}
!cd $BUNDLE_DIR 2>/dev/null || cd .; node scripts/render_colab.mjs --state project_state.json --out output.mp4


In [ ]:
# Step 4: Preview & Download
from IPython.display import Video as IPVideo, display
import pathlib
cand = list(pathlib.Path('.').rglob('output.mp4'))
p = str(cand[0]) if cand else 'output.mp4'
print(f"Preview: {p}")
try:
    display(IPVideo(p, width=640))
except Exception as e:
    print(f"Preview unavailable: {e}")
from google.colab import files
try:
    files.download(p)
except Exception as e:
    print(f"Download trigger failed (use Files panel): {e}")
